# First cosmological analysis: inferring ($\sigma_8, b_g$) from galaxy auto and galaxy x cmb lensing

- The first and second CoLoRe simulation boxes were generated by Laura Casas, Dani L. Jarque run the rest from then on.
- The notebook uses the sources output from CoLoRe, i.e. quasars, to validate the simulation through the auto-spectrum and cross spectrum with CMB lensing.
    - The kappa map (`out_kappa_z000.fits`) can be found in `./data/test_box_x/results/`.
    - The QSO catalogs (`out_srcs_sx_xx.fits`) can be found in `./data/test_box_x/results)/`.
    - The galaxies auto-spectrum $C_{\ell}^{gg}$ and kappa $\times$ galaxies cross-spectrum $C_{\ell}^{\kappa g}$ can be found in `./data/test_box_x/cls/` with the corresponding errorbars, and a logarithmic rebinning run with `./scripts/generate_rebinned_cls.py`.
- The notebook also fits for two parameters in the modelling accounting for CoLoRe finite cell size and a rescaling of the theory bias.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import camb
import healpy as hp

from scipy.optimize import minimize
from scipy.interpolate import PchipInterpolator
from scipy.integrate import simpson
from scipy.constants import c
import zeus, corner
import sys
sys.path.append('../')

from utils.rebinning import hybrid_bin_cls
import json

# functions

## theory

In [ ]:
def load_cosmology(filename):
    with open(filename) as f:
        return json.load(f)

In [ ]:
def get_camb_results(
    cosmo_dict,
    z_pk,
    kmax=100.0,
    lmax=2500,
    nonlinear=False,
):
    """
    Read a cosmological parameter dictionary, extract cosmological parameters,
    and build CAMB results instance
    """

    # Convert to CAMB inputs

    H0 = 100.0 * cosmo_dict["h"]
    ombh2 = cosmo_dict["omega_B"] * cosmo_dict["h"]**2
    omch2 = (cosmo_dict["omega_M"] - cosmo_dict["omega_B"]) * cosmo_dict["h"]**2

    # Initial CAMB params

    pars = camb.CAMBparams()
    pars.set_cosmology(
        H0=H0,
        ombh2=ombh2,
        omch2=omch2,
    )
    pars.set_dark_energy(w=cosmo_dict["w"])
    pars.InitPower.set_params(
        ns=cosmo_dict["ns"],
        As=cosmo_dict["As"]
    )

    # Matter power settings

    pars.set_matter_power(
        redshifts=z_pk,
        kmax=kmax
    )

    # note that to get CAMB's internal calculation accurate at the 1% level at L~2000,
    # need lens_potential_accuracy=2. Increase to 4 for accurate match to the Limber calculation here
    if nonlinear:
        pars.NonLinear = camb.model.NonLinear_pk
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=4, nonlinear=True)
    else:
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=4, nonlinear=False)

    # Final results
    results = camb.get_results(pars)

    return results

In [ ]:
def W_g(z, kernel_context, kernel_args):
    '''
    Galaxy clustering kernel.
    '''

    # jacobian of transforming n(chi)=dN/dchi to n(z)=dN/dz, i.e. dchi/dz 
    c_kms = c * 1e-3
    Hz = kernel_context["Hz"]

    bz_interp = kernel_args["bz_interp"]
    nz_interp = kernel_args["nz_interp"]

    bias = bz_interp(z)
    ndensity = nz_interp(z)

    return bias * ndensity * (Hz / c_kms)

In [ ]:
def W_kappa(z, kernel_context, kernel_args):
    '''
    CMB lensing convergence kernel.
    '''

    # geometry
    a = 1.0 / (1.0 + z)

    # comoving distance to lens and source
    chi = kernel_context["chi"]
    chi_source = kernel_context["chi_source"]

    # cosmological parameters
    H0, om = kernel_context["cosmo_params"]
    c_kms = c * 1e-3

    # kernel prefactor
    W_prefactor = 1.5 * (H0 / c_kms)**2 * om * (chi_source - chi) / chi_source
    W_geom = chi / a

    return W_prefactor * W_geom

## data

In [ ]:
def get_colore_cls(box_filename):
    '''
    Load the Colore kappa power spectrum from a text file
    '''

    # load the data from the .npz file
    data = np.load(box_filename)

    return data['ells'], data['cls'], data['err'], data['nside'], data['shot_noise']

## fitting

In [ ]:
def get_limber_cls_fit(
    params,
    ells,
    z_grid,
    k_grid,
    pk_lin,
    base_gg,
    base_kg,
    pixwin_gg,
    pixwin_kg,
    sigma8_fid,
):
    '''
    Compute the angular qq/kq power spectra using the Limber approximation.
    '''

    # fitting parameters
    Abias, kp_gg, kp_kg, sigma8 = params

    # parameter-dependent smoothing
    smooth_gg = np.exp(-(k_grid / kp_gg) ** 2)
    smooth_kg = np.exp(-(k_grid / kp_kg) ** 2)

    # full integrand
    integrand_gg = base_gg * pk_lin * smooth_gg
    integrand_kg = base_kg * pk_lin * smooth_kg
    growth_factor = (sigma8/sigma8_fid)**2

    # integrate all multipoles simultaneously
    cls_gg = Abias**2 * growth_factor * simpson(
        integrand_gg,
        x=z_grid,
        axis=0,
    )

    cls_kg = Abias * growth_factor * simpson(
        integrand_kg,
        x=z_grid,
        axis=0,
    )
    
    # match the healpix pixel window function with length of cls
    cls_gg = cls_gg[:len(ells)]
    cls_kg = cls_kg[:len(ells)]
    cls_gg_smoothed = cls_gg * (pixwin_gg ** 2)
    cls_kg_smoothed = cls_kg * (pixwin_kg ** 2)

    return {
        "ells": ells,
        "cls_gg": cls_gg_smoothed,
        "cls_kg": cls_kg_smoothed,
    }

In [ ]:
def get_limber_cls_fit_gg(
    params,
    ells,
    z_grid,
    k_grid,
    pk_lin,
    base,
    pixwin,
    sigma8_fid,
):
    '''
    Compute the angular gg power spectra using the Limber approximation.
    '''

    # fitting parameters
    Abias, kp, sigma8 = params

    # parameter-dependent smoothing
    smooth = np.exp(-(k_grid / kp) ** 2)

    # full integrand
    integrand = base * pk_lin * smooth
    growth_factor = (sigma8/sigma8_fid)**2

    # integrate all multipoles simultaneously
    cls = Abias**2 * growth_factor * simpson(
        integrand,
        x=z_grid,
        axis=0,
    )
    
    # match the healpix pixel window function with length of cls
    cls = cls[:len(ells)]
    cls_smoothed = cls * (pixwin ** 2)

    return {
        "ells": ells,
        "cls": cls_smoothed,
    }

In [ ]:
def get_limber_cls_fit_kg(
    params,
    ells,
    z_grid,
    k_grid,
    pk_lin,
    base,
    pixwin,
    sigma8_fid,
):
    '''
    Compute the angular gg power spectra using the Limber approximation.
    '''

    # fitting parameters
    Abias, kp, sigma8 = params

    # parameter-dependent smoothing
    smooth = np.exp(-(k_grid / kp) ** 2)

    # full integrand
    integrand = base * pk_lin * smooth
    growth_factor = (sigma8/sigma8_fid)**2

    # integrate all multipoles simultaneously
    cls = Abias * growth_factor * simpson(
        integrand,
        x=z_grid,
        axis=0,
    )
    
    # match the healpix pixel window function with length of cls
    cls = cls[:len(ells)]
    cls_smoothed = cls * (pixwin ** 2)

    return {
        "ells": ells,
        "cls": cls_smoothed,
    }

In [ ]:
def get_chi2(
    params,
    lmin_fit,
    lmax_fit,
    nbins_fit,
    z_grid,
    k_grid,
    pk_lin,
    base_gg,
    base_kg,
    pixwin_gg,
    pixwin_kg,
    cls_data,
    invcov_mat,
    sigma8_fid,
):
    '''
    Reduced chi-squared for gg and gk fitting.
    '''

    # compute theory
    results = get_limber_cls_fit(
        params,
        ells,
        z_grid,
        k_grid,
        pk_lin,
        base_gg,
        base_kg,
        pixwin_gg,
        pixwin_kg,
        sigma8_fid
    )

    # rebin theory
    _, cls_theory_gg, _ = hybrid_bin_cls(
        results["ells"],
        results["cls_gg"],
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

    _, cls_theory_kg, _ = hybrid_bin_cls(
        results["ells"],
        results["cls_kg"],
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

    # create theory vector
    cls_theory = np.concatenate((cls_theory_gg, cls_theory_kg))

    # remove invalid values
    mask = np.isfinite(cls_data)
    cls_data = cls_data[mask]
    cls_theory = cls_theory[mask]

    # get chi2 statistic
    chi2 = (cls_data - cls_theory) @ invcov_mat @ (cls_data - cls_theory).T
    
    return chi2

In [ ]:
def get_chi2_single(
    params,
    get_limber_cls_function,
    lmin_fit,
    lmax_fit,
    nbins_fit,
    z_grid,
    k_grid,
    pk_lin,
    base,
    pixwin,
    cls_data,
    invcov_mat,
    sigma8_fid,
):
    '''
    Reduced chi-squared for gg and gk fitting.
    '''

    # compute theory
    results = get_limber_cls_function(
        params,
        ells,
        z_grid,
        k_grid,
        pk_lin,
        base,
        pixwin,
        sigma8_fid
    )


    _, cls_theory, _ = hybrid_bin_cls(
        results["ells"],
        results["cls"],
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

    # remove invalid values
    mask = np.isfinite(cls_data)
    cls_data = cls_data[mask]
    cls_theory = cls_theory[mask]

    # get chi2 statistic
    chi2 = (cls_data - cls_theory) @ invcov_mat @ (cls_data - cls_theory).T
    
    return chi2

## sampler

In [ ]:
def log_prior(theta):
    '''
    Flat priors on kp and A_bias.
    '''

    Abias, kp_gg, kp_kg, sigma8 = theta

    if 0 < sigma8 < 10.0 and 0.5 < Abias < 1.5 and 0.01 < kp_gg < 100.0 and 0.01 < kp_kg < 100.0:
        return 0.0

    return -np.inf

In [ ]:
def log_prior_single(theta):
    '''
    Flat priors on kp and A_bias.
    '''

    Abias, kp, sigma8 = theta

    if 0 < sigma8 < 10.0 and 0.5 < Abias < 1.5 and 0.01 < kp < 100.0:
        return 0.0

    return -np.inf

In [ ]:
def log_posterior(
    theta,
    lmin_fit,
    lmax_fit,
    nbins_fit,
    z_grid,
    k_grid,
    pk_lin,
    base_gg,
    base_kg,
    pixwin_gg,
    pixwin_kg,
    cls_data,
    invcov_mat,
    sigma8_fid,
):

    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    chi2 = get_chi2(
        theta,
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_gg,
        base_kg,
        pixwin_gg,
        pixwin_kg,
        cls_data,
        invcov_mat,
        sigma8_fid,
    )

    return lp - 0.5 * chi2

In [ ]:
def log_posterior_single(
    theta,
    get_limber_cls_function,
    lmin_fit,
    lmax_fit,
    nbins_fit,
    z_grid,
    k_grid,
    pk_lin,
    base,
    pixwin,
    cls_data,
    invcov_mat,
    sigma8_fid,
):

    lp = log_prior_single(theta)
    if not np.isfinite(lp):
        return -np.inf

    chi2 = get_chi2_single(
        theta,
        get_limber_cls_function,
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base,
        pixwin,
        cls_data,
        invcov_mat,
        sigma8_fid,
    )

    return lp - 0.5 * chi2

# Fitting sample

## boltzmann parameters

In [ ]:
# set the redshift range for the boltzmann solver
zmin, zmax = 0, 1100
nz = 200
z_pk = zmin + np.exp(np.log(zmax - zmin + 1)*np.linspace(0, 1, nz)) - 1
z_pk = z_pk[::-1] # reverse to have higher redshift first for better interpolation at high z
lmin, lmax = 2, 2500

## measure and compare output n(z)

In [ ]:
# choose the box for which to compare the theory and the CoLoRe Cls
box = 'test_box_4'  # options: 'third_test_box', 'fourth_test_box'
galaxy_catalog = 'lsst_gal'  # options: 'desi_bgs, 'desi_lrg', 'desi_qso', 'lsst_gal'
params_path = f'../data/{box}/cosmo_params.json'

In [ ]:
# load the galaxy catalogs, galaxy bias and redshift distribution, 
z_bz, bz = np.loadtxt(f'../data/colore_files/bz/bz_{galaxy_catalog}.txt', unpack=True)
z_nz_input, nz_input = np.loadtxt(f'../data/colore_files/nz/nz_{galaxy_catalog}.txt', unpack=True)
zbin_width_input = np.diff(z_nz_input)[0]

# load the n(z) histogram output from colore
z_edges, nz_output = np.load(f'../data/{box}/cls/nz_hist_{galaxy_catalog}.npz')['z_edges'], np.load(f'../data/{box}/cls/nz_hist_{galaxy_catalog}.npz')['nz']
zbin_width_output = np.diff(z_edges)[0]

# create n(z) interpolator for the input galaxy bias and redshift distribution
fsky = (4 * np.pi) * (180 / np.pi)**2  # multiply by the total sky area in deg**2 and redshift bin width
nz_input = nz_input * fsky * zbin_width_output
norm = simpson(nz_input, z_nz_input)
nz_interp_input = PchipInterpolator(z_nz_input, nz_input, extrapolate=False)
bz_interp = PchipInterpolator(z_bz, bz, extrapolate=False)

In [ ]:
# full redshift range for interpolation plot
zmin = 0.0
zmax = 3.8
z = np.linspace(zmin, zmax, 1000)

# create n(z) interpolator for the output redshift distributions
z_nz_output = 0.5*(z_edges[:-1] + z_edges[1:])  # get the bins middle points
z_nz_output = np.concatenate(([zmin], z_nz_output, [zmax]))  # add the first and last bin edges to the middle points redshift
z_nz_output_min = nz_input[0]
z_nz_output_max = nz_input[np.abs(z_nz_input - 3.8).argmin()]

nz_output = np.concatenate(([z_nz_output_min], nz_output, [z_nz_output_max]))  # add the first and last bin edges to the middle points n(z)
nz_interp_output = PchipInterpolator(z_nz_output, nz_output, extrapolate=True)

In [ ]:
# define the redshift tomographic bin of the analysis
zmin_bin = 0.0
zmax_bin = 3.8
zbin = np.linspace(zmin_bin, zmax_bin, 1000)
nz_output_bin = nz_output.copy()
nz_output_bin[(z_nz_output <= zmin_bin) | (z_nz_output >= zmax_bin)] = 0  # ensure the interpolator is zero outside the tomographic bin

# create n(z) interpolator for the tomographic bin output redshift distributions
norm = simpson(nz_interp_output(zbin), zbin)
nz_interp_output_bin = PchipInterpolator(z_nz_output, nz_output_bin/norm, extrapolate=True)

In [ ]:
nbins = 400
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4.5), dpi=120, sharex=True, gridspec_kw={'height_ratios':[3,1], 'hspace':0.05})

# Top panel
ax1.bar(z_nz_output, nz_output, width=zbin_width_output, align='center', color='C0', alpha=0.25, label='Output histogram')  # align=edge is needed to match the histogram bins edges with the bars edges
ax1.plot(z_nz_input, nz_interp_input(z_nz_input), 'C1', lw=2.5, label=r'Input $n(z)$')
ax1.plot(z, nz_interp_output(z), 'C3--', lw=2, label=r'Output $n(z)$')
ax1.plot(z, nz_interp_output_bin(z)*norm, 'C2--', lw=2, label=r'Output bin $n(z)$')
ax1.axvline(3.8, color='0.3', ls=':', lw=1.5, label=r'$z_{\rm max}$')
ax1.set_ylabel(r'$\#$ Galaxies')
ax1.set_title(f'{galaxy_catalog.upper()} redshift distribution')
ax1.legend(frameon=True)
ax1.grid(alpha=0.25)

# Bottom panel
ratio = (nz_interp_output_bin(z)*norm - (nz_interp_input(z))) / nz_interp_input(z)
ratio[(nz_interp_input(z) == 0)] = np.nan # set to nan if nz_output is zero
ratio[(nz_interp_output_bin(z) == 0)] = np.nan # set to nan if nz_output is zero
ax2.plot(z, ratio, 'k', lw=1.5)
ax2.axhline(0, color='C3', ls='--', lw=1)
ax2.set_xlim(left=0)
ax2.set_ylim(-1, 1)
ax2.set_xlabel(r'$z$')
ax2.set_ylabel('(Out - In) / In')
ax2.grid(alpha=0.25)

plt.show()

# First analysis: $\sigma_8$ and $b_g$ fit

The plan is to infer $\sigma_8$ by breaking the degeneracy with galaxy bias $b(z)$ by combining galaxy auto and galaxy x cmb lensing spectrums. Marginalizing over $k_p$ for both galaxy auto and galaxy cross. However, we are working with $A_{\rm bias}$ instead of the bias directly, for validation purpouses, so we will be fitting for this parameter instead of the effective bias over the redshift bin $b_{\rm eff}$.

## preparation

In [ ]:
# set the fitting range for the chi2 minimization
lmin_fit = 20
lmax_fit = 1500  # L=1000 by default if NSIDE=1024
nbins_fit = 30   # nbins=30 by default if multipole range from L=20 to L=1000

In [ ]:
# get cosmological parameters
cosmo_dict = load_cosmology(params_path)
z_max = cosmo_dict["z_max"]
sigma8_fid = cosmo_dict['sigma_8']

# get CAMB results
results_lin = get_camb_results(
    cosmo_dict=cosmo_dict,
    z_pk=z_pk,
    kmax=100,
    lmax=lmax,
    nonlinear=False
)

# get matter power spectrum interpolator
pk_interp_lin = results_lin.get_matter_power_interpolator(
    nonlinear=False,
    hubble_units=False,
    k_hunit=False,
    var1='delta_tot',
    var2='delta_tot'
)

z_star = cosmo_dict["z_out"]
chi_star = results_lin.comoving_radial_distance(z_star)


# cosmological parameters needed by kernels
H0 = cosmo_dict["h"] * 100
om = cosmo_dict["omega_M"]

# shared kernel context
chi_source = chi_star
kernel_context = {
    "results": results_lin,
    "chi_source": chi_source,
    "cosmo_params": (H0, om),
}

# precompute the z, chi(z) and H(z) grids for Limber integrand
z_grid = np.linspace(1e-4, z_max, 200)
chi_grid = results_lin.comoving_radial_distance(z_grid)
Hz_grid = results_lin.hubble_parameter(z_grid)

# add to kernel context for use in kernels and Limber integrand
kernel_context["chi"] = chi_grid
kernel_context["Hz"] = Hz_grid

# defined the kernel arguments 
kernel_args={
            "bz_interp": bz_interp,
            "nz_interp": nz_interp_output_bin}

# precompute the kernel grids for Limber integrand
Wk_grid = W_kappa(z_grid, kernel_context, kernel_args)
Wg_grid = W_g(z_grid, kernel_context, kernel_args)

In [ ]:
# load and rebin gg data
box_path_gg = f'../data/{box}/cls/cl_gg_{galaxy_catalog}.npz'
ells_data, cls_gg_data, err_gg_data, nside_gg, shot_noise = get_colore_cls(box_path_gg)
ells_data_rebin, cls_gg_data_rebin, err_gg_data_rebin = hybrid_bin_cls(
        ells_data,
        cls_gg_data,
        err_gg_data,
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

# load and rebin kg data
box_path_kg = f'../data/{box}/cls/cl_kg_{galaxy_catalog}.npz'
ells_data, cls_kg_data, err_kg_data, nside_kg, shot_noise = get_colore_cls(box_path_kg)
ells_data_rebin, cls_kg_data_rebin, err_kg_data_rebin = hybrid_bin_cls(
        ells_data,
        cls_kg_data,
        err_kg_data,
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )

# smooth with healpix pixel window function
pixwin_gg = hp.pixwin(nside_gg)
pixwin_kg = hp.pixwin(nside_kg)

# create data array and covariance matrix
cls_data = np.concatenate((cls_gg_data_rebin, cls_kg_data_rebin))

# compute and rebin covariance off-diagonal terms
fsky = 1
nl_gg = np.ones_like(ells_data) * shot_noise
corr = (2 / ((2*ells_data[:lmax_fit] + 1)*fsky)) * ((cls_gg_data[:lmax_fit] + nl_gg[:lmax_fit]) * cls_kg_data[:lmax_fit])
_, _, corr_rebin = hybrid_bin_cls(
        ells_data[:lmax_fit],
        np.zeros_like(ells_data[:lmax_fit]),
        corr[:lmax_fit],
        nbins=nbins_fit,
        ell_min=lmin_fit,
        ell_max=lmax_fit,
    )
corr_rebin = corr_rebin**2  # hybrid_bin_cls returns the standard deviations, but we need the correlation for the covariance matrix

# construct the covariance matrix between observables
cov_mat = np.block([[np.diag(err_gg_data_rebin**2), np.diag(corr_rebin)],
                    [np.diag(corr_rebin), np.diag(err_kg_data_rebin**2)]])
invcov_mat_gg = np.linalg.inv(np.diag(err_gg_data_rebin**2))
invcov_mat_kg = np.linalg.inv(np.diag(err_kg_data_rebin**2))
invcov_mat = np.linalg.inv(cov_mat)

In [ ]:
# multipoles and cls
ells = np.arange(lmin_fit, lmax_fit + 1)
cls_gg = np.zeros_like(ells, dtype=np.float64)
cls_kg = np.zeros_like(ells, dtype=np.float64)

# geometry
chi = kernel_context["chi"][:, None]        # (Nz,1)
Hz = kernel_context["Hz"][:, None]          # (Nz,1)
c_kms = c * 1e-3

# Limber k-grid (Nz, Nell)
k_grid = (ells[None, :] + 0.5) / chi

# valid CAMB region
kmax = 100.0
mask = (k_grid > 1e-4) & (k_grid < kmax)

# evaluate P(k,z) for all z and ell
pk_lin = np.zeros_like(k_grid)
pk_lin[mask] = pk_interp_lin.P(
    np.broadcast_to(z_grid[:, None], k_grid.shape)[mask],
    k_grid[mask],
    grid=False,
)

# full integrand
base_gg = (c_kms / Hz) * (Wg_grid[:, None] * Wg_grid[:, None]) / chi**2
base_kg = (c_kms / Hz) * (Wk_grid[:, None] * Wg_grid[:, None]) / chi**2

# match the healpix pixel window function with length of cls
lmax_map1 = len(pixwin_gg) - 1
lmax_map2 = len(pixwin_kg) - 1
lmax_min = np.min([lmax_map1, lmax_map2, lmax_fit])
pixwin_gg = pixwin_gg[lmin_fit : (lmax_min + 1)]
pixwin_kg = pixwin_kg[lmin_fit : (lmax_min + 1)]
ells = ells[:(lmax_min + 1 - lmin_fit)]

### likelihood evaluation timer

In [ ]:
x0 = [1.0, 0.5, 0.5, sigma8_fid]  # initial guess for Abias, kp_gg, kp_kg, and sigma8
niter = 1000 # number of iterations for timing the posterior evaluation
t = []

import time

for i in range(niter):
    t0 = time.perf_counter()
    value = log_posterior(
            x0,
            lmin_fit,
            lmax_fit,
            nbins_fit,
            z_grid,
            k_grid,
            pk_lin,
            base_gg,
            base_kg,
            pixwin_gg,
            pixwin_kg,
            cls_data,
            invcov_mat,
            sigma8_fid,
        )
    t.append(time.perf_counter() - t0)

print(f"posterior evaluation: {np.mean(t):.3f} s")

## minimizer

In [ ]:
# perform the chi2 minimization
x0 = [1.0, 0.5, 0.5, sigma8_fid]  # initial guess for Abias, kp_gg, kp_kg, and sigma8
bounds = [(0.5, 1.5), (0.01, 1000.0), (0.01, 1000.0), (0.01, 5)]  # bounds for Abias, kp_gg, kp_kg, and sigma8
result = minimize(
    get_chi2,
    x0=x0,
    bounds=bounds,
    method="L-BFGS-B",
    args=(
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_gg,
        base_kg,
        pixwin_gg,
        pixwin_kg,
        cls_data,
        invcov_mat,
        sigma8_fid,
    )
)

# extract best-fit parameters, covariance and chi2
best_Abias, best_kp_gg, best_kp_kg, best_sigma8  = result.x
cov = result.hess_inv.todense()
errors = np.sqrt(np.diag(cov))
chi2_best = result.fun

In [ ]:
print(f"Best-fit sigma8: {best_sigma8:.2f} +/- {errors[-1]:.2f}")
print(f"Best-fit A_bias: {best_Abias:.2f} +/- {errors[0]:.2f}")
print(f"Best-fit kp_gg   : {best_kp_gg:.2f} +/- {errors[1]:.2f} 1/Mpc")
print(f"Best-fit kp_kg   : {best_kp_kg:.2f} +/- {errors[2]:.2f} 1/Mpc")
print(f"Reduced chi2 at best-fit: {chi2_best:.2f}")

## sampler

In [ ]:
# scipy best fit as starting point for MCMC
theta0 = result.x

# initialize the MCMC
ndim = len(theta0)
nwalkers = 32  # number of walkers for MCMC, normally 3 times the number of dimensions

# initialize walkers around best fit
start = theta0 + 1e-1 * np.random.randn(nwalkers, ndim)

In [ ]:
# initialize the sampler
sampler = zeus.EnsembleSampler(
    nwalkers,
    ndim,
    log_posterior,
    args=(
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_gg,
        base_kg,
        pixwin_gg,
        pixwin_kg,
        cls_data,
        invcov_mat,
        sigma8_fid,
    )
)

# run the MCMC
nsteps = 1500
sampler.run_mcmc(start, nsteps, progress=True) 

In [ ]:
# save the chains to a file
raw_chain = sampler.get_chain()

map_idx = np.argmax(sampler.get_log_prob(flat=True))
map_estimate = sampler.get_chain(flat=True)[map_idx]

truths = np.array([1.0, 0.0, 0.0, sigma8_fid])  # Assuming the true value of A_bias is 1.0 for the purpose of this example
np.savez(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}.npz", 
         chain=raw_chain, nsteps=nsteps, nwalkers=nwalkers, ndim=ndim, map_estimate=map_estimate, bestfit_values=result.x, true_values=truths)

## visualization

In [ ]:
# load the chains from the file to visualize the results and clean the samples
box = 'test_box_4'  # options: 'third_test_box', 'fourth_test_box'
galaxy_catalog = 'lsst_gal'  # options: 'desi_bgs', 'desi_elg', 'desi_lrg', 'desi_qso', 'lsst_gal'
chain_file = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps1500.npz", allow_pickle=True)
chain = chain_file['chain']
nsteps = chain_file['nsteps']
nwalkers = chain_file['nwalkers']
ndim = chain_file['ndim']
map_estimate = chain_file['map_estimate']
bestfit_values = chain_file['bestfit_values']
truths = chain_file['true_values']

In [ ]:
fig, axes = plt.subplots(int(ndim), figsize=(10, 8), sharex=True)
axis_labels = [
    r"$A_{\rm bias}$",
    r"$k_{p,gg}\,[\mathrm{Mpc}^{-1}]$",
    r"$k_{p,\kappa g}\,[\mathrm{Mpc}^{-1}]$",
    r"$\sigma_8$",
]

for i in range(ndim):
    for walker in range(chain.shape[1]):
        ax = axes[i]
        # Plot the path of all walkers across all steps for this dimension
        ax.plot(chain[:, walker, i], alpha=0.5)
    ax.set_xlim(0, len(chain))
    ax.set_ylabel(axis_labels[i])
    axes[-1].set_xlabel('steps')

In [ ]:
burnin = 425
deleted_walkers = []  # indices of walkers to delete

clean_chain = np.delete(chain, obj=deleted_walkers, axis=1)  # discard the first walker as it is not well initialized
clean_chain = clean_chain[burnin:, : , :]  # discard the first burnin steps as burn-in
clean_samples = clean_chain.reshape(-1, clean_chain.shape[-1])  # reshape to 2D array for corner plot

np.savez(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}.npz", 
         chain=chain, clean_samples=clean_samples, nsteps=nsteps, burnin=burnin, nwalkers=nwalkers, deleted_walkers=deleted_walkers, ndim=ndim, 
         map_estimate=map_estimate, bestfit_values=bestfit_values, true_values=truths)

In [ ]:
fig, axes = plt.subplots(int(ndim), figsize=(10, 8), sharex=True)
axis_labels = [
    r"$A_{\rm bias}$",
    r"$k_{p,gg} \ [\mathrm{Mpc}^{-1}]$",
    r"$k_{p,\kappa g} \ [\mathrm{Mpc}^{-1}]$",
    r"$\sigma_8$",
]

for i in range(ndim):
    for walker in range(clean_chain.shape[1]):
        ax = axes[i]
        # Plot the path of all walkers across all steps for this dimension
        ax.plot(clean_chain[:, walker, i], alpha=0.5)
    ax.set_xlim(0, clean_chain.shape[0])
    ax.set_ylabel(axis_labels[i])
    axes[-1].set_xlabel('steps')

## plotter

In [ ]:
# choose the box for which to compare the theory and the CoLoRe Cls
box = 'test_box_4'  # options: 'third_test_box', 'fourth_test_box'
galaxy_catalog = 'lsst_gal'  # options: 'desi_bgs', 'desi_elg', 'desi_lrg', 'desi_qso', 'lsst_gal'
chain_file = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}.npz", allow_pickle=True)
chain_file_gg = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_gg.npz", allow_pickle=True)
chain_file_kg = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_kg.npz", allow_pickle=True)
samples = chain_file['clean_samples']
truths = chain_file['true_values']
map_estimate = chain_file['map_estimate']
samples_gg = chain_file_gg['clean_samples'] 
samples_kg = chain_file_kg['clean_samples']

In [ ]:
# 1σ and 2σ enclosed probabilities for a 2D Gaussian
levels = [
    1.0 - np.exp(-0.5),  # 1σ ≈ 0.393
    1.0 - np.exp(-2.0),  # 2σ ≈ 0.865
]

bins=75

fig = corner.corner(
    samples,
    labels=axis_labels,

    # Only show truths for sigma8 and Abias
    truths=truths,

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:blue",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
)

q = np.percentile(samples, [16, 50, 84], axis=0)

titles = [
    rf"$A_{{\rm bias}} = {q[1,0]:.3f}^{{+{q[2,0]-q[1,0]:.3f}}}_{{-{q[1,0]-q[0,0]:.3f}}}$",
    rf"$k_{{p,gg}} = {q[1,1]:.3f}^{{+{q[2,1]-q[1,1]:.3f}}}_{{-{q[1,1]-q[0,1]:.3f}}}\,"
    rf"\mathrm{{Mpc}}^{{-1}}$",
    rf"$k_{{p,\kappa g}} = {q[1,2]:.3f}^{{+{q[2,2]-q[1,2]:.3f}}}_{{-{q[1,2]-q[0,2]:.3f}}}\,"
    rf"\mathrm{{Mpc}}^{{-1}}$",
    rf"$\sigma_8 = {q[1,3]:.3f}^{{+{q[2,3]-q[1,3]:.3f}}}_{{-{q[1,3]-q[0,3]:.3f}}}$",
]

axes = np.asarray(fig.axes).reshape(4, 4)

for i, title in enumerate(titles):
    axes[i, i].set_title(title, fontsize=12)

fig.suptitle(
    rf"Constraints from $C_\ell^{{gg}}$ and $C_\ell^{{\kappa g }}$ for {galaxy_catalog.upper()}",
    fontsize=18,
    y=1.05,
)

plt.savefig(f"../data/{box}/chains/corner_plot_{galaxy_catalog}_nsteps{nsteps}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig = corner.corner(
    samples[:, [0, -1]],  # Only show Abias and sigma8
    labels=[axis_labels[0], axis_labels[-1]],

    # Only show truths for sigma8 and Abias
    truths=[truths[0], truths[-1]],

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:blue",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="C3",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
)

fig.suptitle(
    rf"Constraints from $C_\ell^{{gg}}$ and $C_\ell^{{\kappa g }}$ for {galaxy_catalog.upper()}",
    fontsize=15,
    y=1.05,
)

axes = np.asarray(fig.axes).reshape(2, 2)
axes[1, 0].scatter(map_estimate[0], map_estimate[-1], color="blue", marker="*", s=100, label="MAP estimate", zorder=5)

fig.legend(bbox_to_anchor=(0.54, 0.54), fontsize=8)

plt.savefig(f"../data/{box}/chains/corner_plot_{galaxy_catalog}_nsteps{nsteps}_zoom.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig = corner.corner(
    samples_gg[:, [0, -1]],  # Only show Abias and sigma8
    labels=[axis_labels[0], axis_labels[-1]],

    # Only show truths for sigma8 and Abias
    truths=[truths[0], truths[-1]],

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:green",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
)

corner.corner(
    samples_kg[:, [0, -1]],  # Only show Abias and sigma8
    labels=[axis_labels[0], axis_labels[-1]],

    # Only show truths for sigma8 and Abias
    truths=[truths[0], truths[-1]],

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:orange",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
    fig=fig
)

corner.corner(
    samples[:, [0, -1]],  # Only show Abias and sigma8
    labels=[axis_labels[0], axis_labels[-1]],

    # Only show truths for sigma8 and Abias
    truths=[truths[0], truths[-1]],

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:blue",

    # Pass transparency directly to the underlying 2D filler
    contourf_kwargs={"alpha": 0.0}, 

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
    fig=fig
)

fig.suptitle(
    rf"Constraints from $C_\ell^{{gg}}$ and $C_\ell^{{\kappa g }}$ for {galaxy_catalog.upper()}",
    fontsize=15,
    y=1.05,
)

axes = np.asarray(fig.axes).reshape(2, 2)
axes[1, 0].scatter(map_estimate[0], map_estimate[-1], color="blue", marker="*", s=100, label="MAP estimate", zorder=5)

fig.legend(bbox_to_anchor=(0.54, 0.54), fontsize=8)

f = 0.02  # zoom factor for the axes limits
axes[0, 0].set_xlim(map_estimate[0]*(1-f), map_estimate[0]*(1+f))
axes[1, 0].set_xlim(map_estimate[0]*(1-f), map_estimate[0]*(1+f))
axes[1, 0].set_ylim(map_estimate[-1]*(1-f), map_estimate[-1]*(1+f))
axes[1, 1].set_xlim(map_estimate[-1]*(1-f), map_estimate[-1]*(1+f))

plt.savefig(f"../data/{box}/chains/corner_plot_{galaxy_catalog}_nsteps{nsteps}_zoom2.png", dpi=300, bbox_inches="tight")
plt.show()

## minimizer-sampler-plotter-single

In [ ]:
# perform the chi2 minimization
x0 = [1.0, 0.5,  sigma8_fid]  # initial guess for Abias, kp, and sigma8
bounds = [(0.5, 1.5), (0.01, 1000.0),  (0.01, 5)]  # bounds for Abias, kp, and sigma8
result_gg = minimize(
    get_chi2_single,
    x0=x0,
    bounds=bounds,
    method="L-BFGS-B",
    args=(
        get_limber_cls_fit_gg,
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_gg,
        pixwin_gg,
        cls_gg_data_rebin,
        invcov_mat_gg,
        sigma8_fid,
    )
)

# extract best-fit parameters, covariance and chi2
best_Abias, best_kp, best_sigma8  = result_gg.x
cov = result_gg.hess_inv.todense()
errors = np.sqrt(np.diag(cov))
chi2_best = result_gg.fun

In [ ]:
print(f"Best-fit sigma8: {best_sigma8:.2f} +/- {errors[-1]:.2f}")
print(f"Best-fit A_bias: {best_Abias:.2f} +/- {errors[0]:.2f}")
print(f"Best-fit kp   : {best_kp:.2f} +/- {errors[1]:.2f} 1/Mpc")
print(f"Reduced chi2 at best-fit: {chi2_best:.2f}")

In [ ]:
# perform the chi2 minimization
x0 = [1.0, 0.5,  sigma8_fid]  # initial guess for Abias, kp, and sigma8
bounds = [(0.5, 1.5), (0.01, 1000.0),  (0.01, 5)]  # bounds for Abias, kp, and sigma8
result_kg = minimize(
    get_chi2_single,
    x0=x0,
    bounds=bounds,
    method="L-BFGS-B",
    args=(
        get_limber_cls_fit_kg,
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_kg,
        pixwin_kg,
        cls_kg_data_rebin,
        invcov_mat_kg,
        sigma8_fid,
    )
)

# extract best-fit parameters, covariance and chi2
best_Abias, best_kp, best_sigma8  = result_kg.x
cov = result_kg.hess_inv.todense()
errors = np.sqrt(np.diag(cov))
chi2_best = result_kg.fun

In [ ]:
print(f"Best-fit sigma8: {best_sigma8:.2f} +/- {errors[-1]:.2f}")
print(f"Best-fit A_bias: {best_Abias:.2f} +/- {errors[0]:.2f}")
print(f"Best-fit kp   : {best_kp:.2f} +/- {errors[1]:.2f} 1/Mpc")
print(f"Reduced chi2 at best-fit: {chi2_best:.2f}")

In [ ]:
# scipy best fit as starting point for MCMC
theta0_gg = result_gg.x
theta0_kg = result_kg.x 

# initialize the MCMC
ndim = len(theta0_gg)
nwalkers = 24  # number of walkers for MCMC, normally 3 times the number of dimensions

# initialize walkers around best fit
start_gg = theta0_gg + 1e-1 * np.random.randn(nwalkers, ndim)
start_kg = theta0_kg + 1e-1 * np.random.randn(nwalkers, ndim)

In [ ]:
# initialize the sampler
sampler_gg = zeus.EnsembleSampler(
    nwalkers,
    ndim,
    log_posterior_single,
    args=(
        get_limber_cls_fit_gg,
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_gg,
        pixwin_gg,
        cls_gg_data_rebin,
        invcov_mat_gg,
        sigma8_fid,
    )
)

# initialize the sampler
sampler_kg = zeus.EnsembleSampler(
    nwalkers,
    ndim,
    log_posterior_single,
    args=(
        get_limber_cls_fit_kg,
        lmin_fit,
        lmax_fit,
        nbins_fit,
        z_grid,
        k_grid,
        pk_lin,
        base_kg,
        pixwin_kg,
        cls_kg_data_rebin,
        invcov_mat_kg,
        sigma8_fid,
    )
)

In [ ]:
# run the MCMC
nsteps = 1500
sampler_gg.run_mcmc(start_gg, nsteps, progress=True)
sampler_kg.run_mcmc(start_kg, nsteps, progress=True)

In [ ]:
# save the chains to a file
raw_chain_gg = sampler_gg.get_chain()
raw_chain_kg = sampler_kg.get_chain()

map_idx_gg = np.argmax(sampler_gg.get_log_prob(flat=True))
map_estimate_gg = sampler_gg.get_chain(flat=True)[map_idx_gg]
map_idx_kg = np.argmax(sampler_kg.get_log_prob(flat=True))
map_estimate_kg = sampler_kg.get_chain(flat=True)[map_idx_kg]

truths = np.array([1.0, 0.0, sigma8_fid])  # Assuming the true value of A_bias is 1.0 for the purpose of this example
np.savez(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_gg.npz", 
         chain=raw_chain_gg, nsteps=nsteps, nwalkers=nwalkers, ndim=ndim, 
         map_estimate=map_estimate_gg, bestfit_values=result_gg.x, true_values=truths)
np.savez(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_kg.npz", 
         chain=raw_chain_kg, nsteps=nsteps, nwalkers=nwalkers, ndim=ndim, 
         map_estimate=map_estimate_kg, bestfit_values=result_kg.x, true_values=truths)

### visualization

In [ ]:
# load the chains from the file to visualize the results and clean the samples
box = 'test_box_4'  # options: 'third_test_box', 'fourth_test_box'
galaxy_catalog = 'lsst_gal'  # options: 'desi_bgs', 'desi_elg', 'desi_lrg', 'desi_qso', 'lsst_gal'
chain_file_gg = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_gg.npz", allow_pickle=True)
chain_file_kg = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_kg.npz", allow_pickle=True)
chain_gg = chain_file_gg['chain']  # discard the last 1000 samples as burn-in
chain_kg = chain_file_kg['chain']  # discard the last 1000 samples as burn-in
nsteps = chain_file_gg['nsteps']
nwalkers = chain_file_gg['nwalkers']
ndim = chain_file_gg['ndim']
map_estimate_gg = chain_file_gg['map_estimate']
map_estimate_kg = chain_file_kg['map_estimate']
bestfit_values_gg = chain_file_gg['bestfit_values']
bestfit_values_kg = chain_file_kg['bestfit_values']
truths = chain_file_gg['true_values']

In [ ]:
fig, axes = plt.subplots(int(ndim), figsize=(10, 8), sharex=True)

axis_labels_single = [
    r"$A_{\rm bias}$",
    r"$k_{p} \ [\mathrm{Mpc}^{-1}]$",
    r"$\sigma_8$",
]

for i in range(ndim):
    for walker in range(chain_gg.shape[1]):
        ax = axes[i]
        # Plot the path of all walkers across all steps for this dimension
        ax.plot(chain_gg[:, walker, i], c='C0', alpha=0.5)
        ax.plot(chain_kg[:, walker, i], c='C1', alpha=0.5)
    ax.set_xlim(0, len(chain_gg))
    ax.set_ylabel(axis_labels_single[i])
    axes[-1].set_xlabel('steps')

In [ ]:
chain_gg[500, :, 1]

In [ ]:
burnin = 250
deleted_walkers_gg = [7]  # indices of walkers to delete for gg
deleted_walkers_kg = [0]  # indices of walkers to delete for kg

clean_chain_gg = np.delete(chain_gg, obj=deleted_walkers_gg, axis=1)  # discard the first walker as it is not well initialized
clean_chain_kg = np.delete(chain_kg, obj=deleted_walkers_kg, axis=1)  # discard the first walker as it is not well initialized
clean_chain_gg = clean_chain_gg[burnin:, : , :]  # discard the first 250 steps as burn-in
clean_chain_kg = clean_chain_kg[burnin:, : , :]  # discard the first 250 steps as burn-in
clean_samples_gg = clean_chain_gg.reshape(-1, clean_chain_gg.shape[-1])  # reshape to 2D array for corner plot
clean_samples_kg = clean_chain_kg.reshape(-1, clean_chain_kg.shape[-1])  # reshape to 2D array for corner plot

np.savez(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_gg.npz", 
         chain=chain_gg, clean_samples=clean_samples_gg, nsteps=nsteps, burnin=burnin, nwalkers=nwalkers, deleted_walkers=deleted_walkers_gg, ndim=ndim, 
         map_estimate=map_estimate_gg, bestfit_values=bestfit_values_gg, true_values=truths)
np.savez(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_kg.npz", 
         chain=chain_kg, clean_samples=clean_samples_kg, nsteps=nsteps, burnin=burnin, nwalkers=nwalkers, deleted_walkers=deleted_walkers_kg, ndim=ndim, 
         map_estimate=map_estimate_kg, bestfit_values=bestfit_values_kg, true_values=truths)

In [ ]:
fig, axes = plt.subplots(int(ndim), figsize=(10, 8), sharex=True)

for i in range(ndim):
    for walker in range(clean_chain_gg.shape[1]):
        ax = axes[i]
        # Plot the path of all walkers across all steps for this dimension
        ax.plot(clean_chain_gg[:, walker, i], c='C0', alpha=0.5)
        ax.plot(clean_chain_kg[:, walker, i], c='C1', alpha=0.5)
    ax.set_xlim(0, clean_chain_gg.shape[0])
    ax.set_ylabel(axis_labels_single[i])
    axes[-1].set_xlabel('steps')

### plotter

In [ ]:
# choose the box for which to compare the theory and the CoLoRe Cls
box = 'test_box_4'  # options: 'third_test_box', 'fourth_test_box'
galaxy_catalog = 'lsst_gal'  # options: 'desi_bgs', 'desi_elg', 'desi_lrg', 'desi_qso', 'lsst_gal'
chain_file_gg = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_gg.npz", allow_pickle=True)
chain_file_kg = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}_kg.npz", allow_pickle=True)
chain_file = np.load(f"../data/{box}/chains/chain_{galaxy_catalog}_nsteps{nsteps}.npz", allow_pickle=True)
samples = chain_file['clean_samples']
samples_gg = chain_file_gg['clean_samples'] 
samples_kg = chain_file_kg['clean_samples']
truths = chain_file_gg['true_values']

In [ ]:
# 1σ and 2σ enclosed probabilities for a 2D Gaussian
levels = [
    1.0 - np.exp(-0.5),  # 1σ ≈ 0.393
    1.0 - np.exp(-2.0),  # 2σ ≈ 0.865
]

bins=75

fig = corner.corner(
    samples_gg,
    labels=axis_labels_single,

    # Only show truths for sigma8 and Abias
    truths=truths,

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:blue",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
)

corner.corner(
    samples_kg,
    labels=axis_labels_single,

    # Only show truths for sigma8 and Abias
    truths=truths,

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:orange",

    # General appearance
    smooth=None,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
    fig=fig
)


q_gg = np.percentile(samples_gg, [16, 50, 84], axis=0)
q_kg = np.percentile(samples_kg, [16, 50, 84], axis=0)

titles = [
    rf"$A_{{\rm bias}} = ({q_gg[1,0]:.3f}^{{+{q_gg[2,0]-q_gg[1,0]:.3f}}}_{{-{q_gg[1,0]-q_gg[0,0]:.3f}}},\ "
    rf'{q_kg[1,0]:.3f}^{{+{q_kg[2,0]-q_kg[1,0]:.3f}}}_{{-{q_kg[1,0]-q_kg[0,0]:.3f}}})$',
    rf"$\qquad \qquad \qquad k_p = ({q_gg[1,1]:.3f}^{{+{q_gg[2,1]-q_gg[1,1]:.3f}}}_{{-{q_gg[1,1]-q_gg[0,1]:.3f}}}, \ "
    rf"{q_kg[1,1]:.3f}^{{+{q_kg[2,1]-q_kg[1,1]:.3f}}}_{{-{q_kg[1,1]-q_kg[0,1]:.3f}}}) \ "
    rf"\mathrm{{Mpc}}^{{-1}}$",
    rf"$\qquad \quad \sigma_8 = ({q_gg[1,2]:.3f}^{{+{q_gg[2,2]-q_gg[1,2]:.3f}}}_{{-{q_gg[1,2]-q_gg[0,2]:.3f}}}, \ "
    rf"{q_kg[1,2]:.3f}^{{+{q_kg[2,2]-q_kg[1,2]:.3f}}}_{{-{q_kg[1,2]-q_kg[0,2]:.3f}}})$",
]

axes = np.asarray(fig.axes).reshape(3, 3)

for i, title in enumerate(titles):
    axes[i, i].set_title(title, fontsize=12)

fig.suptitle(
    rf"Constraints from $C_\ell^{{gg}}$ and $C_\ell^{{\kappa g }}$ for {galaxy_catalog.upper()}",
    fontsize=18,
    y=1.05,
)

plt.savefig(f"../data/{box}/chains/corner_plot_{galaxy_catalog}_nsteps{nsteps}_single.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig = corner.corner(
    samples_gg[:, [0, -1]],  # Only show Abias and sigma8
    labels=[axis_labels_single[0], axis_labels_single[-1]],

    # Only show truths for sigma8 and Abias
    truths=[truths[0], truths[-1]],

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:blue",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
)

corner.corner(
    samples_kg[:, [0, -1]],  # Only show Abias and sigma8
    labels=[axis_labels_single[0], axis_labels_single[-1]],

    # Only show truths for sigma8 and Abias
    truths=[truths[0], truths[-1]],

    # Contours
    levels=levels,
    fill_contours=True,
    plot_contours=True,
    color="tab:orange",

    # General appearance
    smooth=1.0,
    bins=bins,
    show_titles=True,
    title_fmt=".3f",
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 12},

    # Samples / contours
    plot_datapoints=False,
    plot_density=False,

    # Truth lines
    truth_color="tab:red",

    # Histograms
    hist_kwargs={
        "density": True,
        "linewidth": 1.5,
    },

    # Contour borders
    contour_kwargs={
        "linewidths": 1.5,
    },
    label="hola",
    fig=fig
)

q_gg = np.percentile(samples_gg, [16, 50, 84], axis=0)
q_kg = np.percentile(samples_kg, [16, 50, 84], axis=0)

titles = [
    rf"$A_{{\rm bias}} = ({q_gg[1,0]:.3f}^{{+{q_gg[2,0]-q_gg[1,0]:.3f}}}_{{-{q_gg[1,0]-q_gg[0,0]:.3f}}},\ "
    rf'{q_kg[1,0]:.3f}^{{+{q_kg[2,0]-q_kg[1,0]:.3f}}}_{{-{q_kg[1,0]-q_kg[0,0]:.3f}}})$',
    rf"$\qquad \quad \sigma_8 = ({q_gg[1,2]:.3f}^{{+{q_gg[2,2]-q_gg[1,2]:.3f}}}_{{-{q_gg[1,2]-q_gg[0,2]:.3f}}},\ "
    rf"{q_kg[1,2]:.3f}^{{+{q_kg[2,2]-q_kg[1,2]:.3f}}}_{{-{q_kg[1,2]-q_kg[0,2]:.3f}}})$",
]

axes = np.asarray(fig.axes).reshape(2, 2)

for i, title in enumerate(titles):
    axes[i, i].set_title(title, fontsize=12)

axes = np.asarray(fig.axes).reshape(2, 2)
axes[1, 0].scatter(map_estimate_gg[0], map_estimate_gg[-1], color="blue", marker="*", s=100, label="MAP gg estimate", zorder=5)
axes[1, 0].scatter(map_estimate_kg[0], map_estimate_kg[-1], color="red", marker="*", s=100, label="MAP kg estimate", zorder=5)

fig.suptitle(
    rf"Constraints from $C_\ell^{{gg}}$ and $C_\ell^{{\kappa g }}$ for {galaxy_catalog.upper()}",
    fontsize=18,
    y=1.05,
)
fig.legend(bbox_to_anchor=(0.54, 0.54), fontsize=8)

plt.savefig(f"../data/{box}/chains/corner_plot_{galaxy_catalog}_nsteps{nsteps}_single_zoom.png", dpi=300, bbox_inches="tight")
plt.show()